In [7]:
from mini_llm.inference import InferenceEngine, GenerationConfig, GenerationResult
from mini_llm.model import ModelConfig

In [8]:
device="mps"
lm_path = "checkpoints/latest.pt"

config = GenerationConfig(
    model=ModelConfig(),
    max_new_tokens=128,
    temperature=0.7,
    top_p=0.9,
    top_k=40,
    do_sample=True,
    eos_token_id=50256,
    lm_path=lm_path,
    device=device,
    dtype="fp32"
)

In [ ]:
engine = InferenceEngine(config)

text = "i like computer"
result = engine.generate_naive(text)

In [12]:
engine.tokenizer.decode(result.generated_ids)

' Webb Sagaonymous melarnaev etiquette reinvestIndividualGh Jane Middle international Gard clinch billed Sul shift Stats Supporting GAM BitFour omissionev Santanaauto jack Sac FnMurraymis month unfairaram drivewayyetForcetermination installing fossilsistics Lenn booze unveilingingle affiliatedcano<< euro Graveyardpertyological undraftedBlogcars excessive nefarious weren Fail STOR brutallyuum dancedigen Sagling multiploutput|| switches tower Sanskrit starosis magazines io Cosby Conce scrape Astittees protest carrots simulator Und speaks Aff constitute chats inflation dig dedicatedabus GPUs storing EPA injuryFortenz broch pancakes Jord encomp announ Briefthen shadowyijkictiveamiliarfurt repo explaining glossyBarn looted ravico Challenges 445 BCnosticppeDataumo overarchingos 326'

In [2]:
from mini_llm.kv_cache import PagedLayerKVCache, PagedLayerState, PagedKVCache
import torch

device = 'mps'   # 当前环境无 CUDA，先用 CPU 实验
num_heads = 16
head_dim = 64
num_blocks = 4
cache = PagedLayerKVCache(num_heads=num_heads,num_blocks=num_blocks, block_size=8, head_dim=head_dim, device=device, dtype=torch.float32)

# 模拟多次 append
for step in range(5):
    # 模拟新 token 的 K/V (batch=1)
    k_new = torch.randn(1, num_heads, 3, head_dim, device=device)
    v_new = torch.randn(1, num_heads, 3, head_dim, device=device)
    
    cache.append(k_new, v_new)
    state = cache.get_paged_state()
    
    print(f"Step {step}: block_table = {state.block_table}")
    print(f"  length={state.length}, used_blocks={len(state.block_table)}\n")

Step 0: block_table = [3]
  length=3, used_blocks=1

Step 1: block_table = [3]
  length=6, used_blocks=1

Step 2: block_table = [3, 2]
  length=9, used_blocks=2

Step 3: block_table = [3, 2]
  length=12, used_blocks=2

Step 4: block_table = [3, 2]
  length=15, used_blocks=2



In [2]:
import torch
from torch.nn.attention.flex_attention import flex_attention, create_block_mask

# ====================== 最小的 FlexAttention 示例 ======================

def causal_mask(b, h, q_idx, kv_idx):
    """最简单的 causal mask"""
    return q_idx >= kv_idx

device = "cpu"
print(f"Using device: {device}")

# 参数
B = 1
H = 4
Q_LEN = 5
KV_LEN = 8
D = 32

# 随机生成 Q, K, V
q = torch.randn(B, H, Q_LEN, D, device=device)
k = torch.randn(B, H, KV_LEN, D, device=device)
v = torch.randn(B, H, KV_LEN, D, device=device)

# 创建 BlockMask
block_mask = create_block_mask(
    causal_mask, 
    B=B, 
    H=H, 
    Q_LEN=Q_LEN, 
    KV_LEN=KV_LEN, 
    device=device
)

# 调用 FlexAttention
output = flex_attention(
    query=q,
    key=k,
    value=v,
    block_mask=block_mask
)

print(f"Q shape:      {q.shape}")
print(f"K shape:      {k.shape}")
print(f"Output shape: {output.shape}")
print("✅ FlexAttention 调用成功！")

Using device: cpu
Q shape:      torch.Size([1, 4, 5, 32])
K shape:      torch.Size([1, 4, 8, 32])
Output shape: torch.Size([1, 4, 5, 32])
✅ FlexAttention 调用成功！
